# DocuMind - loan contracts: detector (YOLO) from the labelled boxes to the checked key terms

Same idea as the receipt and cheque notebooks: the notebook only calls the package (`documind/`), run it top to bottom with the kernel of your GPU environment.

```
data/processed/contracts_yolo (prepared)  ->  [documind.training.train_contracts]  ->  models/contracts_yolo.pt
picture  ->  [documind.inference.contracts: detector + OCR on each box + checks]  ->  parties, amount, rate, duration, monthly payment, dates, IBAN, signatures
```
Make the data once, in a terminal (the notebook never rebuilds it):
```
python scripts/generate_contracts.py --n 10000 --out data/raw/contracts
python -m documind.data.prepare_contracts          # then LOOK at reports/contracts_preview/
```
Classes (read from `data/processed/contracts_yolo/data.yaml`): 19, from `title` and `contract_number` to `iban`, `signature_lender`, `signature_borrower` and `stamp`.
A contract is a full A4 page of running text: **several boxes of one class on a page are normal** (the lender's name is printed twice, a value that does not fit at the end of a line wraps onto the next one and gets one box per line). The reading step puts them back together.
The contract, cheque and receipt models are independent files: training one never touches the others.

In [ ]:
import json, os, sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image
# Make the project root importable when the notebook is opened from a subfolder.
ROOT = Path.cwd()
while not (ROOT / "documind").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if (ROOT / "documind").is_dir():
    os.chdir(ROOT)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    print(f"project root: {ROOT}")
else:
    raise FileNotFoundError(
        "Could not find the project root containing 'documind/'. "
        "Open the notebook from the repo root or install the project with: %pip install -e ."
    )

from documind.config import MODELS_DIR, PROCESSED_DIR, RAW_DIR, REPORTS_DIR

If a package is missing (first run): `pip install -r requirements-receipts.txt` in a terminal with the GPU environment activated (Ultralytics brings PyTorch: big download). Contracts use the same requirements as receipts and cheques.

## GPU check (run this first)
Ultralytics only uses the GPU if PyTorch is a **CUDA build** and the NVIDIA driver works. `pip install ultralytics` often installs the CPU-only PyTorch on Windows, so check before training.

If the check below stops with an error, fix it in a terminal with `ai_env` activated, then **restart the kernel**:
```
pip uninstall -y torch torchvision torchaudio
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
```

In [ ]:
import torch

REQUIRE_GPU = True   # True = stop here if no GPU is found (instead of silently training for hours on the CPU)

print("python executable :", sys.executable)          # must be inside your ai_env
print("torch             :", torch.__version__)       # ending with +cpu = CPU-only build
print("torch CUDA build  :", torch.version.cuda)      # None = CPU-only build
print("cuda available    :", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"gpu               : {props.name} ({props.total_memory / 1024**3:.1f} GB VRAM)")
    DEVICE = 0
else:
    DEVICE = "cpu"
    msg = ("No usable GPU for PyTorch. Check, in this order:\n"
           "  1. torch version ends with '+cpu' / CUDA build is None -> reinstall the CUDA build (cell above)\n"
           "  2. python executable is not your ai_env -> Kernel > Change kernel\n"
           "  3. 'nvidia-smi' fails in a terminal -> update the NVIDIA driver")
    if REQUIRE_GPU:
        raise RuntimeError(msg)
    print("WARNING (training will run on the CPU):\n" + msg)

## Settings

In [ ]:
from documind.data.prepare_contracts import CONTRACT_NAMES
from documind.data.prepare_receipts import read_names_from_yaml
from documind.training import train_contracts, train_receipts   # contracts reuse the receipt trainer with their own Profile (paths, imgsz, augmentation)

PROFILE = train_contracts.profile()
CLASS_NAMES = read_names_from_yaml(PROFILE.data_yaml) or CONTRACT_NAMES   # class ids 0..N-1, read from the prepared data
print(len(CLASS_NAMES), "classes:", CLASS_NAMES)

MODEL = PROFILE.model     # yolo11n (fastest) < yolo11s < yolo11m (most accurate, more GPU memory)
EPOCHS = PROFILE.epochs   # 60; 100 = all epochs (also set PATIENCE = 100)
TARGET = 0.90             # score to reach on val AND test (mAP50), checked after the evaluation
IMGSZ = PROFILE.imgsz     # 1280: an A4 page with 20 px text. 1536 gives the small values (rate, duration, ID) more pixels (then BATCH = 2)
BATCH = PROFILE.batch     # 4 fits an 8 GB GPU at 1280 with yolo11s; lower it (2) on 'CUDA out of memory'; -1 = automatic
PATIENCE = PROFILE.patience   # stop after this many epochs without progress (best.pt is still the best epoch)
NAME = "contracts"
# DEVICE (0 = first GPU, or "cpu") was set by the GPU check above
WORKERS = max(2, min(8, os.cpu_count() or 2))   # data-loader processes
CACHE = False             # "ram" or "disk" decode every picture once (needs several GB): faster epochs
FRACTION = 1.0            # 0.3 = train on 30 % of the pictures: quick run to compare settings

## Look at the data
The data was prepared in an earlier run (`python -m documind.data.prepare_contracts`). If the boxes in the pictures do not sit on the right things, **stop here**. Note the `lender_name` count: about twice the number of pages, because the name is printed twice.

In [ ]:
DATA_ROOT = PROFILE.data_yaml.parent
DATA_YAML = PROFILE.data_yaml
assert DATA_YAML.exists(), "prepared data not found: run  python -m documind.data.prepare_contracts  once in a terminal"

counts = {}
for split in ("train", "val", "test"):
    n = [0] * len(CLASS_NAMES)
    for p in (DATA_ROOT / split / "labels").glob("*.txt"):
        for line in p.read_text().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            try:
                cls_id = int(float(parts[0]))
            except ValueError:
                continue
            if 0 <= cls_id < len(n):
                n[cls_id] += 1
    counts[split] = n
df = pd.DataFrame(counts, index=CLASS_NAMES)
display(df)
df.plot(kind="bar", figsize=(11, 4), title="boxes per class and split"); plt.tight_layout(); plt.show()
prev = sorted((REPORTS_DIR / "contracts_preview").glob("*.png"))[:2]
if prev:
    fig, ax = plt.subplots(len(prev), 1, figsize=(7, 9.5 * len(prev)))
    for a, p in zip(ax if len(prev) > 1 else [ax], prev):
        a.imshow(Image.open(p)); a.set_title(p.stem[:40], fontsize=8); a.axis("off")
    plt.show()

## STEP 2 - train
Augmentation chosen for contracts (see `documind/training/train_contracts.py`): **no mosaic** (it would shrink four pages into one and make the 20 px text even smaller), no flips (a mirrored page is not a contract), a small rotation (scans are tilted by a degree or two), brightness changes.

The same trainer as the receipts and cheques (`train_receipts.train`), given the contract `Profile`: its paths (`data/processed/contracts_yolo`, `models/contracts_yolo.pt`, `reports/contracts_runs`), its augmentation and its messages. The first lines of the training log must show `CUDA:0 (NVIDIA GeForce RTX 3080 ...)`.

**Speed check while an epoch runs:** open Task Manager > Performance > GPU (look at the *CUDA* or *Compute* graph) or run `nvidia-smi -l 1` in a terminal.
Below about 70 % the GPU is waiting for the data loader: raise `WORKERS` or set `CACHE = "ram"` / `"disk"`.
Above 90 % it is the GPU: use `MODEL = "yolo11n.pt"` or `IMGSZ = 800` and compare with `FRACTION = 0.3` runs first.

In [ ]:
import inspect
import os

import torch

def _accepts(fn, arg):
    params = inspect.signature(fn).parameters
    return arg in params or any(p.kind is inspect.Parameter.VAR_KEYWORD for p in params.values())

if "DEVICE" not in globals():
    DEVICE = 0 if torch.cuda.is_available() else "cpu"

train_kwargs = dict(
    data=PROFILE.data_yaml,
    model=MODEL,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    name=NAME,
    workers=WORKERS,
    cache=CACHE,
    fraction=FRACTION,
    profile=PROFILE,
)
if _accepts(train_receipts.train, "device"):
    train_kwargs["device"] = DEVICE

weights = train_receipts.train(**train_kwargs)
print("best weights:", weights)

In [ ]:
run_dir = PROFILE.runs_dir / NAME
for f in ("results.png", "confusion_matrix_normalized.png"):
    path = run_dir / f
    if path.exists():
        plt.figure(figsize=(11, 6))
        plt.imshow(Image.open(path))
        plt.axis("off")
        plt.title(f)
        plt.show()

## Evaluate on val AND test
`test` was never used for training or for choosing the best epoch. Read the **per-class** rows: a good average can hide a class that never works (the small values `borrower_id`, `interest_rate`, `duration` and the wrapped ones `iban`, `amount_words` are the usual suspects).

`found_rate`: for a class that has one box per page (`amount`, `interest_rate` ...) = is the most confident box of that class on the labelled box (IoU >= 0.5)? For a class that can have several (`lender_name`, `borrower_address`, `iban`) = the share of labelled boxes found. `mAP50` is the strict one.

**Be careful with what a high score means:** the contracts come from `generate_contracts.py`, and val / test come from the same generator as train. A score close to 1.0 says the model learned the generated layouts, not that it reads every real contract. Try real contracts (STEP 3) before trusting it.

In [ ]:
eval_kwargs = dict(imgsz=IMGSZ, profile=PROFILE)
if _accepts(train_receipts.evaluate, "device"):
    eval_kwargs["device"] = DEVICE
rows = train_receipts.evaluate(weights, PROFILE.data_yaml, **eval_kwargs)
display(pd.DataFrame(rows).round(3))
# did we reach the target (val AND test)?  mAP50 is the strict number, found_rate the one closer to what the user sees
for metric in ("mAP50", "found_rate"):
    print(f"--- {metric}")
    train_receipts.print_verdict(train_receipts.check_target(rows, TARGET, metric), PROFILE.hints, PROFILE.reached_note)

## If the score is below the target
The labels of `generate_contracts.py` are exact by construction, so there is no label audit. In order of effect: look at the weakest class above and at the confusion matrix; more pixels (`IMGSZ = 1536`, `BATCH = 2`); all the epochs (`EPOCHS = 100`, `PATIENCE = 100`); a bigger model (`MODEL = "yolo11m.pt"`, `BATCH = 2`); more pages (`python scripts/generate_contracts.py --n 20000 --seed 1`, then prepare again).

## See the predictions on test pictures

In [ ]:
from ultralytics import YOLO
model = YOLO(str(weights))
test_imgs = sorted((DATA_ROOT / "test" / "images").glob("*.jpg"))[:2]
res = model.predict(source=[str(p) for p in test_imgs], imgsz=IMGSZ, conf=0.25, device=DEVICE, verbose=False)
fig, ax = plt.subplots(len(res), 1, figsize=(7, 9.5 * len(res)))
for a, r in zip(ax if len(res) > 1 else [ax], res):
    a.imshow(r.plot()[..., ::-1]); a.axis("off")
plt.show()

## STEP 3 - from boxes to checked key terms
The detector says **where** each term is; the OCR reads **what** is written there (`documind/inference/contracts.py`; the trained OCR of `train_ocr` if `models/ocr_trocr` exists, else Tesseract). The boxes of one field are put back in reading order (the page tilt is measured and removed first) and joined, then the values are parsed and **checked**:

- amount in figures = amount in words (the French words are converted back to a number, millions included)
- monthly payment = the loan formula for the amount, the rate and the duration (within 1 %: insurance and fees make real contracts differ a little)
- the dates are real, the first payment is after the contract date, the signing date is not before it
- the IBAN has the length of its country (MA 28, FR 27), both signatures were found

A failed check is a warning, never a silent correction. A warning on a generated contract is usually an OCR slip on a long digit string: that is exactly what the checks are for.

In [ ]:
from documind.inference.contracts import extract_contract_bytes
for p in test_imgs[:2]:
    r = extract_contract_bytes(p.read_bytes(), p.name, model=model)
    print(p.name)
    print(json.dumps({"row": r["row"], "checks": r["checks"], "warnings": [w["message"] for w in r["warnings"]]}, ensure_ascii=False, indent=2)[:2600], "\n")

### How often does each check pass? (20 test pages)
The generated contracts are all consistent, so a failed check here is an OCR mistake, not a bad contract. `not checked` = a value could not be read as a number.

In [ ]:
from collections import Counter
all_imgs = sorted((DATA_ROOT / "test" / "images").glob("*.jpg"))[:20]
results = [extract_contract_bytes(p.read_bytes(), p.name, model=model) for p in all_imgs]
print("amount in figures vs in words:", dict(Counter(r["row"]["AMOUNT_CHECK"] for r in results)))
print("monthly payment vs loan formula:", dict(Counter(r["row"]["PAYMENT_CHECK"] for r in results)))
print("both signatures found         :", dict(Counter((r["row"]["SIGNED_LENDER"], r["row"]["SIGNED_BORROWER"]) for r in results)))
print("warnings                      :", dict(Counter(w["code"] for r in results for w in r["warnings"])))

### Notes
- The model is saved in `models/contracts_yolo.pt`, the scores in `reports/contracts_report.json` (both are git-ignored like the rest).
- **GPU memory:** with `yolo11s`, `IMGSZ = 1280`, `BATCH = 4` it should fit 8 GB; on `CUDA out of memory` lower `BATCH` to 2. `IMGSZ = 1536` needs `BATCH = 2`.
- **Licence:** Ultralytics YOLO is AGPL-3.0 (a commercial licence exists): fine for a university project, read it before a commercial use.
- **OCR:** to improve the reading of the values themselves, train the OCR on the contract crops too: `python -m documind.data.prepare_ocr` (contracts are a source) then `python -m documind.training.train_ocr`.
- Command-line version of the same steps: `python -m documind.training.train_contracts`, `python -m documind.inference.contracts contract.jpg`.